# 03 Retrieval

`src.retrieval` compare dense / bm25 / hybrid retriever on hit, recall, MRR, latency metrics

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")

## Sample question

In [ ]:
from src.retrieval.factory import build_retriever
from src.common.config import get_settings

settings = get_settings()
query = "가계부실위험지수(HDRI)는 무엇인가요?"

retrievers = {
    "dense": build_retriever(mode="dense", k=5),
    "bm25": build_retriever(mode="bm25", k=5),
    "hybrid": build_retriever(mode="hybrid", k=5),
}

In [ ]:
for name, retriever in retrievers.items():
    docs = retriever.invoke(query)
    print(f"[{name}] top-{len(docs)}")
    for i, doc in enumerate(docs, start=1):
        print(f"{i}. {doc.metadata.get('chunk_id')} | {doc.metadata.get('term')}")

## Run Evaluation

In [ ]:
from src.evaluation.pipeline import run_retriever_comparison_evaluation

settings = get_settings()
detail_path = settings.eval_data_dir / "retriever_comparison_detail.csv"
summary_path = settings.eval_data_dir / "retriever_comparison_summary.csv"

detail_df, summary_df = run_retriever_comparison_evaluation(
    eval_csv_path=settings.eval_data_dir / "golden_testset_v2.csv",
    chunk_json_path=settings.default_chunk_json_path,
    output_csv_path=detail_path,
    output_summary_csv_path=summary_path,
    k=5,
)

print(f"detail saved: {detail_path}")
print(f"summary saved: {summary_path}")
summary_df

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

try:
    settings
except NameError:
    from src.common.config import get_settings
    settings = get_settings()

summary_path = settings.eval_data_dir / "retriever_comparison_summary.csv"
summary_df = pd.read_csv(summary_path)

def retriever_label(row):
    if row["mode"] == "bm25":
        return "bm25"
    provider = row["dense_provider"]
    model = row["dense_model_name"].replace("text-embedding-3-small", "text-emb-3-small")
    model = model.replace("BAAI/", "")
    return f"{row['mode']}\n{provider}\n{model}"

plot_df = summary_df.copy()
plot_df["retriever"] = plot_df.apply(retriever_label, axis=1)
plot_df = plot_df.sort_values(["recall", "mrr", "hit"], ascending=False).reset_index(drop=True)

display_cols = ["experiment", "hit", "recall", "mrr", "avg_query_latency_sec"]
display(
    plot_df[display_cols]
    .style.format({
        "hit": "{:.3f}",
        "recall": "{:.3f}",
        "mrr": "{:.3f}",
        "avg_query_latency_sec": "{:.3f}s",
    })
    .background_gradient(subset=["hit", "recall", "mrr"], cmap="YlGn")
    .background_gradient(subset=["avg_query_latency_sec"], cmap="YlOrRd_r")
)

colors = {
    "bm25": "#4C78A8",
    "dense": "#F58518",
    "hybrid": "#54A24B",
}
bar_colors = plot_df["mode"].map(colors)

fig, axes = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True)
quality_metrics = ["hit", "recall", "mrr"]

for ax, metric in zip(axes.flat[:3], quality_metrics):
    ax.bar(plot_df["retriever"], plot_df[metric], color=bar_colors)
    ax.set_title(metric.upper())
    ax.set_ylim(0.85, 1.01)
    ax.grid(axis="y", alpha=0.25)
    ax.tick_params(axis="x", rotation=0, labelsize=9)
    for i, value in enumerate(plot_df[metric]):
        ax.text(i, value + 0.004, f"{value:.3f}", ha="center", va="bottom", fontsize=8)

latency_ax = axes.flat[3]
latency_ax.bar(plot_df["retriever"], plot_df["avg_query_latency_sec"], color=bar_colors)
latency_ax.set_title("Average Query Latency")
latency_ax.set_ylabel("seconds")
latency_ax.grid(axis="y", alpha=0.25)
latency_ax.tick_params(axis="x", rotation=0, labelsize=9)
for i, value in enumerate(plot_df["avg_query_latency_sec"]):
    latency_ax.text(i, value + 0.01, f"{value:.3f}s", ha="center", va="bottom", fontsize=8)

fig.suptitle("Retriever Comparison Summary", fontsize=16)
plt.show()

fig, ax = plt.subplots(figsize=(9, 6))
for mode, mode_df in plot_df.groupby("mode"):
    ax.scatter(
        mode_df["avg_query_latency_sec"],
        mode_df["recall"],
        s=mode_df["mrr"] * 260,
        color=colors.get(mode, "#999999"),
        alpha=0.8,
        label=mode,
    )
    for _, row in mode_df.iterrows():
        ax.annotate(row["experiment"], (row["avg_query_latency_sec"], row["recall"]), xytext=(6, 4), textcoords="offset points", fontsize=8)

ax.set_title("Recall vs Latency (bubble size = MRR)")
ax.set_xlabel("Average query latency (sec)")
ax.set_ylabel("Recall")
ax.grid(alpha=0.25)
ax.legend(title="mode")
plt.show()